# RoBERTa-base SQuAD2 — DIMER E2E extractive question-answering fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/roberta-squad2-question-answering-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/roberta-squad2-question-answering-pipeline/blob/main/tutorials/roberta_question_answering_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-deepset%2Froberta--base--squad2-ffcc4d?style=flat)](https://huggingface.co/deepset/roberta-base-squad2) [![Upstream](https://img.shields.io/badge/Upstream-deepset--ai%2Fhaystack-181717?style=flat&logo=github&logoColor=white)](https://github.com/deepset-ai/haystack) [![arXiv](https://img.shields.io/badge/arXiv-1907.11692-b31b1b.svg)](https://arxiv.org/abs/1907.11692)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** extractive question answering with the SQuAD 2.0 unanswerable case and bounded supervised fine-tuning of the last encoder blocks and the span head on a gold-span dataset, using the pinned `deepset/roberta-base-squad2` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/roberta_question_answering_pipeline/`, at revision `8743bd03e717`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `adc3b06f79f797d1c575d5479d6f5efe54a9e3b4` (~498 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `deepset/roberta-base-squad2` snapshot (safetensors, 496 MB), fetches the digest-pinned AdversarialQA corpus from the project site (9.0 MB, no credential), filters the dRoBERTa subset and cuts it by article into 1,000 / 200 / 400 training, validation and test questions, drops any record the pipeline's token ceilings would refuse, answers three authored questions through the inference contract with an input manifest and a rejection probe, scores the frozen model on the test split with exact-match and F1 beside the always-null and lexical-overlap baselines, runs a bounded fine-tuning of the last two encoder blocks and the span head on the training questions with validation-F1 epoch selection, scores the held-out split again, answers new questions with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify answer parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about six minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own question–answer pairs as a JSON array of `{{id, question, context, answers}}` records, a SQuAD-format JSON file, a JSONL file, or a CSV with columns `id, question, context, answer_text, answer_start`. They pass through the same validation, seeded passage-disjoint split, fit check, baselines, fine-tuning, held-out evaluation, inference, artifact export and reload-parity cells as the AdversarialQA sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`deepset/roberta-base-squad2` is the 124 M-parameter `roberta-base` encoder (Liu et al., 2019) with a span head, fine-tuned by deepset on SQuAD 2.0 — question-answer pairs **including unanswerable questions** — and published under **CC-BY-4.0** (attribution: deepset; the weights are redistributed unmodified). It is an **extractive reader**: it can only copy one contiguous span out of the passage you give it, and it can decide that the passage contains no answer. At inference the encoder reads `<s> question </s></s> context </s>` once and emits a start and an end logit per token; the carried module then applies the **upstream null-vs-span rule** (`DECISION_RULE`): softmax the start and end logits over the context tokens plus `<s>`, take the best span as the argmax of `P_start(i)·P_end(j)` with `i ≤ j < i + max_answer_tokens`, score "no answer" as `P_start(<s>)·P_end(<s>)`, and return the **empty answer** when the null score wins.

What this notebook adds to inference is **adaptation with gold spans**. The dataset is real and deliberately hard: AdversarialQA v1.0 (Bartolo et al., TACL 2020, CC BY-SA 3.0) — SQuAD-style questions over Wikipedia passages written by annotators who could see a reader's predictions and kept only the questions it got wrong. The `dRoBERTa` subset was collected against a RoBERTa reader, so the frozen `roberta-base-squad2` scores far below its SQuAD 2.0 dev figures here (the build record measured F1 12.4 on the test split, with the null answer returned for more than half of the answerable questions), and the fine-tuning question is whether a small in-distribution adaptation of the last two encoder blocks and the span head recovers ground on held-out articles. Two metrics are implemented in the carried `metrics.py` (corpus **exact-match** and **F1** with the official SQuAD 2.0 normalisation), and two **non-neural baselines** — always-null and lexical overlap — show where a reader that does nothing, or only counts shared words, sits. Nothing here is a quality claim about your domain: it is one seeded split of one corpus.

**Snapshot note:** the pinned revision ships **no `tokenizer.json`** (a 7-file manifest), so `AutoTokenizer` builds `RobertaTokenizerFast` from `vocab.json` and `merges.txt`; the weights are `model.safetensors` — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those seven files before the tokenizer or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, dataset and metrics modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned adversarial QA corpus and validate and split it by article without leakage; drop the records the token ceilings would refuse rather than truncating them; answer through the public API with an explicit `max_answer_tokens` and read `answer`, `score`, `no_answer_score`, `best_span_score` and `answerable` correctly (products of softmax masses, not calibrated probabilities); score the frozen model against gold spans beside two non-neural baselines and read what the null-answer rate says; run a bounded fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on an article-disjoint test split; answer new questions; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** generative or abstractive answering, yes/no or counting questions, multi-hop reasoning across sentences, retrieval over many documents (this reader takes one passage the caller supplies), passages over `MAX_CONTEXT_TOKENS` (refused, not windowed — chunk them yourself), languages other than English, batching, top-k alternative answers, a tuned no-answer threshold offset, full-model or embedding fine-tuning, training on unanswerable questions beyond the SQuAD 2.0 `<s>` convention, and any claim that an AdversarialQA split stands in for your domain. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is adequate but not fast: the build record measured 6 s to load and digest-verify the 498 MB snapshot, 20 s to answer the 400-question test split, and about 3.5 minutes per epoch of fine-tuning the last two encoder blocks and the span head on 1,000 questions (validation scoring included). The pinned `torch==2.14.0` install and the 496 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what an encoder-only Transformer is; what span extraction means; why a reader can return a fluent-looking wrong span, or an empty answer for a question the passage does answer; what exact-match and F1 measure and why neither is a human judgement.
- **Data contract:** records are `{{id, question, context, answers}}` — a question, its passage and a list of `{{text, answer_start}}` gold spans (empty for unanswerable), each `text` found verbatim at its offset; questions at most `MAX_QUESTION_CHARS` (500) characters and `MAX_QUESTION_TOKENS` (64) BPE tokens, passages at most `MAX_CONTEXT_CHARS` (3,000) characters and `MAX_CONTEXT_TOKENS` (384) tokens, enforced by rejecting, never by truncating or windowing; ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a dataset needs 8..20,000 records; every question on the same passage lands in the same split so a test passage is never trained on. BYOD accepts a JSON array, SQuAD-format JSON, JSONL or CSV in that shape.
- **Validation is structural, not semantic:** nothing checks that a question is answerable from its passage beyond the gold text being present at its offset, or that a gold span is the *best* answer — a mislabelled corpus is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an internal document set with its question log is exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches one pinned object (`aqa_v1.0.zip`, 9,018,914 bytes, SHA-256 `f4f3c232…`) from `adversarialqa.github.io` over HTTPS, refused on any mismatch before it is read; only the `3_droberta/train.json` and `3_droberta/dev.json` members are read, and the corpus is CC BY-SA 3.0 (Bartolo et al., 2020).
- **External access:** the Hugging Face Hub only, to fetch the pinned `deepset/roberta-base-squad2` snapshot (~498 MB in total) at revision `adc3b06f79f7…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'roberta-squad2-question-answering-pipeline',
    'repository_revision': '8743bd03e717f9cecb5d91797f234c7ce60de43d',
    'embedded_module': 'src/roberta_question_answering_pipeline/pipeline.py',
    'embedded_modules': ['src/roberta_question_answering_pipeline/metrics.py', 'src/roberta_question_answering_pipeline/pipeline.py', 'src/roberta_question_answering_pipeline/samples.py'],
    'module_sha256': '58b0f07eb2a25e8cef9005813b0ff7109814038fede842bb62692ac0f6caa612',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/roberta_question_answering_pipeline/` @ `8743bd03e717`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/roberta_question_answering_pipeline/metrics.py`

In [ ]:
"""SQuAD-style per-pair helpers, corpus-level extractive-QA metrics and two non-neural baselines.

The per-pair helpers (`normalize_answer`, `exact_match`, `f1`) follow the official SQuAD 2.0 evaluation
script's normalisation (re-exported by `pipeline.py`). This module averages them over a dataset and adds two
baselines a fine-tuned reader must beat: **always-null** (the empty answer for every question — it scores
exactly the unanswerable fraction of the set) and **lexical overlap** (return the passage sentence sharing
the most normalised tokens with the question — a bag-of-words reader with no model).
"""

from __future__ import annotations

import re
import string
from collections import Counter
from collections.abc import Mapping, Sequence
from typing import Any

_SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")


def normalize_answer(text: str) -> str:
    """SQuAD-style normalisation: lower-case, drop punctuation and the articles a/an/the, collapse spaces."""
    text = "".join(ch for ch in text.lower() if ch not in set(string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(prediction: str, gold_answers: Sequence[str]) -> float:
    """1.0 if the normalised prediction equals any normalised gold answer (an empty gold list means
    unanswerable and matches only the empty prediction), else 0.0."""
    golds = [normalize_answer(g) for g in gold_answers] or [""]
    return 1.0 if normalize_answer(prediction) in golds else 0.0


def f1(prediction: str, gold_answers: Sequence[str]) -> float:
    """Best token-overlap F1 against any gold answer, SQuAD 2.0 style: for an unanswerable gold (empty
    list or empty string) the score is 1.0 only when the prediction is empty too."""
    golds = list(gold_answers) or [""]
    pred_tokens = normalize_answer(prediction).split()
    best = 0.0
    for gold in golds:
        gold_tokens = normalize_answer(gold).split()
        if not pred_tokens or not gold_tokens:
            score = float(pred_tokens == gold_tokens)
        else:
            common = sum((Counter(pred_tokens) & Counter(gold_tokens)).values())
            if common == 0:
                score = 0.0
            else:
                precision, recall = common / len(pred_tokens), common / len(gold_tokens)
                score = 2 * precision * recall / (precision + recall)
        best = max(best, score)
    return best


METRIC_DEFINITIONS = {
    "exact_match": (
        "mean over questions of 1[normalised prediction equals any normalised gold]; SQuAD normalisation "
        "(lower-case, strip punctuation and a/an/the, collapse spaces); an unanswerable gold matches only "
        "the empty prediction; reported in percent"
    ),
    "f1": (
        "mean over questions of the best token-overlap F1 against any gold (1.0 for an empty prediction on "
        "an unanswerable gold, else 0.0 when either side is empty); reported in percent"
    ),
}


def qa_metrics(predictions: Sequence[str], golds: Sequence[Sequence[str]]) -> dict[str, Any]:
    """Corpus exact-match and F1 in percent over parallel predictions and SQuAD-style gold lists."""
    if len(predictions) != len(golds):
        raise ValueError(f"{len(predictions)} predictions but {len(golds)} gold lists")
    if not predictions:
        raise ValueError("no predictions to score")
    em = [exact_match(p, g) for p, g in zip(predictions, golds, strict=True)]
    f1s = [f1(p, g) for p, g in zip(predictions, golds, strict=True)]
    return {
        "n": len(predictions),
        "exact_match": 100.0 * sum(em) / len(em),
        "f1": 100.0 * sum(f1s) / len(f1s),
        "answered_rate": 100.0 * sum(1 for p in predictions if p) / len(predictions),
        "unanswerable_gold": sum(1 for g in golds if not g),
        "definitions": dict(METRIC_DEFINITIONS),
    }


def _golds(records: Sequence[Mapping[str, Any]]) -> list[list[str]]:
    return [[a["text"] for a in r["answers"]] for r in records]


def null_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """The empty answer for every question: scores the unanswerable fraction of the set and nothing else."""
    result = qa_metrics([""] * len(records), _golds(records))
    result["baseline"] = "always the empty (no-answer) prediction"
    return result


def lexical_overlap_answer(question: str, context: str) -> str:
    """The passage sentence with the most normalised tokens in common with the question (first on ties)."""
    q_tokens = set(normalize_answer(question).split())
    best, best_overlap = "", -1
    for sentence in _SENTENCE_RE.split(context.strip()):
        overlap = len(q_tokens & set(normalize_answer(sentence).split()))
        if overlap > best_overlap:
            best, best_overlap = sentence, overlap
    return best


def lexical_overlap_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """A bag-of-words reader with no model: the best-overlapping sentence as the whole answer."""
    predictions = [lexical_overlap_answer(r["question"], r["context"]) for r in records]
    result = qa_metrics(predictions, _golds(records))
    result["baseline"] = "lexical overlap (passage sentence sharing the most question tokens)"
    return result

**Module 2/3:** `src/roberta_question_answering_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Extractive question answering with the pinned ``deepset/roberta-base-squad2`` checkpoint.

The class loads weights only from a digest-verified local snapshot (``weights/roberta-base-squad2/``) or,
when explicitly allowed, from the Hugging Face Hub at the pinned revision. One task method, ``answer``:
a question and a context in, one context span (or the SQuAD 2.0 empty "no answer") out, decided with
the same null-vs-span rule the upstream ``transformers`` question-answering pipeline applies.

The adaptation contract (``evaluate``, ``adapt``, ``save_artifact``, ``from_artifact``) fine-tunes the last
encoder blocks plus the span head on a validated ``{id, question, context, answers}`` dataset with
validation-F1 epoch selection and exports the trained tensors as a safetensors adapter bound to the pinned
base weights. The inference contract above is unchanged by it.
"""

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np

# standalone rewrite (build_notebook.py): `from .metrics import exact_match, f1, normalize_answer  # noqa: F401 -- re-exported public helpers` removed — names are kernel globals defined by the carried modules

MODEL_ID = "deepset/roberta-base-squad2"
MODEL_REVISION = "adc3b06f79f797d1c575d5479d6f5efe54a9e3b4"
MODEL_LICENSE = "cc-by-4.0"
MODEL_KEY = "roberta-base-squad2"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. Hard ceilings, no sliding window: a question/context pair that does not fit is rejected, never
# split or truncated. 64 and 384 are the upstream fine-tuning settings (`max_query_length=64`,
# `max_seq_len=386` with `doc_stride=128` in the pinned README); 64 + 384 + 4 special tokens = 452 <= the
# 512-token `model_max_length` in the pinned tokenizer_config.json, so an accepted pair fits in one pass.
MAX_QUESTION_TOKENS = 64
MAX_CONTEXT_TOKENS = 384
MAX_QUESTION_CHARS = 500  # pre-tokenisation guard; ~4 chars per BPE token on English text
MAX_CONTEXT_CHARS = 3_000
MAX_ANSWER_TOKENS = 64  # ceiling on the span length a caller may request
DEFAULT_MAX_ANSWER_TOKENS = 15  # the upstream pipeline's `max_answer_len` default
NULL_MASK_VALUE = -10000.0  # the upstream pipeline's fill for non-context positions before the softmax
DECISION_RULE = (
    "softmax start and end logits over the context tokens plus <s>; no_answer_score = P_start(<s>) * "
    "P_end(<s>); best span = argmax P_start(i) * P_end(j) over i <= j < i + max_answer_tokens inside the "
    "context; the empty answer is returned when no_answer_score > best span score (upstream "
    "handle_impossible_answer=True rule); scores are products of softmax masses, not calibrated probabilities"
)
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "ac5db66fdcfecb400345d09787b71009d60805ef9883451071669cf951b5e2c7"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 124_056_578
ENCODER_LAYERS = 12  # config.json num_hidden_layers
DEFAULT_TRAINABLE_ENCODER_LAYERS = 2  # the last two encoder blocks plus the span head (14,177,282 parameters)
MAX_EVAL_RECORDS = 2_000
MAX_RECORDS_FIT = 20_000  # check_fit accepts a whole dataset before it is split
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.roberta-base-squad2.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one (question, context) pair of non-empty str; the answer is a span of the context or empty",
    "question_chars": [1, MAX_QUESTION_CHARS],
    "context_chars": [1, MAX_CONTEXT_CHARS],
    "question_tokens": [1, MAX_QUESTION_TOKENS],
    "context_tokens": [1, MAX_CONTEXT_TOKENS],
    "max_answer_tokens": [1, MAX_ANSWER_TOKENS],
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "byte-level BPE (vocab.json + merges.txt, no lower-casing) of the pair "
        "<s> question </s></s> context </s>; hard ceilings, no sliding window: a question over "
        "MAX_QUESTION_TOKENS or a context over MAX_CONTEXT_TOKENS is rejected with a ValueError naming the "
        "count, never truncated or chunked"
    ),
}


def _check_text(text: Any, name: str, max_chars: int) -> str:
    if not isinstance(text, str):
        raise TypeError(f"{name} must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError(f"{name} is empty")
    if len(text) > max_chars:
        raise ValueError(f"{name} has {len(text)} chars; ceiling is MAX_{name.upper()}_CHARS={max_chars}")
    return text


def _check_inputs(question: Any, context: Any, max_answer_tokens: Any) -> tuple[str, str]:
    """Raise TypeError/ValueError naming the first violated ceiling; return (question, context).

    The token ceilings are not checked here because they need the loaded tokenizer;
    ``_check_token_counts`` applies them inside the pipeline once the counts are known.
    """
    question = _check_text(question, "question", MAX_QUESTION_CHARS)
    context = _check_text(context, "context", MAX_CONTEXT_CHARS)
    if isinstance(max_answer_tokens, bool) or not isinstance(max_answer_tokens, int):
        raise TypeError("max_answer_tokens must be an int")
    if not 1 <= max_answer_tokens <= MAX_ANSWER_TOKENS:
        raise ValueError(
            f"max_answer_tokens must be between 1 and {MAX_ANSWER_TOKENS}, got {max_answer_tokens}"
        )
    return question, context


def _check_token_counts(n_question: int, n_context: int) -> tuple[int, int]:
    """The token ceilings, applied once the tokenizer has counted (no special tokens)."""
    if n_question > MAX_QUESTION_TOKENS:
        raise ValueError(
            f"question is {n_question} tokens; ceiling is MAX_QUESTION_TOKENS={MAX_QUESTION_TOKENS}"
        )
    if n_context > MAX_CONTEXT_TOKENS:
        raise ValueError(f"context is {n_context} tokens; ceiling is MAX_CONTEXT_TOKENS={MAX_CONTEXT_TOKENS}")
    return n_question, n_context


def validate_inputs(
    questions: Sequence[str],
    contexts: Sequence[str],
    *,
    max_answer_tokens: int = DEFAULT_MAX_ANSWER_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-pair observations, verdict).

    Rejection is reported by raising exactly as ``answer`` would: both route through ``_check_inputs``.
    ``answer`` takes one pair per call, so ``questions``/``contexts`` are the parallel batch the notebook
    will loop over. The token ceilings (``MAX_QUESTION_TOKENS``, ``MAX_CONTEXT_TOKENS``) need the loaded
    tokenizer and are enforced inside ``answer``, which reports both counts in every result.
    """
    for label, value in (("questions", questions), ("contexts", contexts)):
        if isinstance(value, str | bytes) or not isinstance(value, Sequence):
            raise TypeError(f"{label} must be a sequence of str, not a single string")
    if not questions:
        raise ValueError("questions must hold at least one item")
    if len(questions) != len(contexts):
        raise ValueError("questions and contexts must have the same length")
    checked = [_check_inputs(q, c, max_answer_tokens) for q, c in zip(questions, contexts, strict=True)]
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per pair")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"pair{i:02d}", "question_chars": len(q), "context_chars": len(c)}
            for i, (q, c) in enumerate(checked)
        ],
        "max_answer_tokens": max_answer_tokens,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], gold_answers: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report for one ``answer`` result.

    With ``gold_answers`` (the SQuAD convention: a list of acceptable strings; an empty list means the
    question is unanswerable) the report carries ``exact_match`` and ``f1`` for that single pair and the
    verdict ``sample-sanity`` — one pair is a plumbing check, not an accuracy. Without gold the verdict is
    ``not-measurable``.
    """
    prediction = str(result.get("answer", ""))
    supplied = gold_answers is not None
    metrics = []
    if supplied:
        metrics = [
            {
                "id": "exact_match",
                "value": exact_match(prediction, gold_answers),
                "estimation": "single pair",
            },
            {"id": "f1", "value": f1(prediction, gold_answers), "estimation": "single pair"},
        ]
    return {
        "task": "extractive question answering with the SQuAD 2.0 unanswerable case",
        "decision_rule": DECISION_RULE,
        "sample_kind": sample_kind,
        "n_pairs": 1,
        "answerable": bool(prediction),
        "metrics": metrics,
        "baselines": [],
        "verdict": "sample-sanity" if supplied else "not-measurable",
        "reason": (
            "exact_match and f1 are computed for one (question, context, gold) triple with the repository's "
            "SQuAD-style normalisation; a single pair states no dispersion and is not an accuracy"
            if supplied
            else "no gold answer was supplied, so exact_match and f1 cannot be computed"
        ),
        "needs": (
            "gold answer spans (SQuAD 2.0 format: a list of acceptable strings per question, empty for "
            "unanswerable) over enough held-out pairs from the deployment domain to state a dispersion, "
            "scored with the repository's exact_match and f1 helpers"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class RoBERTaQuestionAnsweringPipeline:
    """``_runner(question, context)`` -> ``(start_logits (T,), end_logits (T,), offsets (T, 2),
    context_mask (T,))`` for the encoded pair; ``_count_tokens(text)`` -> BPE token count without special
    tokens. Both injectable so tests run offline."""

    _runner: Callable[[str, str], tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]
    _count_tokens: Callable[[str], int]
    device: str = "cpu"
    source: str = "injected"
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> RoBERTaQuestionAnsweringPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), dict(local_files_only=True), "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, dict(revision=MODEL_REVISION), "hf-hub"
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoTokenizer, RobertaForQuestionAnswering

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(location, trust_remote_code=False, **kwargs)
        model = RobertaForQuestionAnswering.from_pretrained(
            location, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def count_tokens(text: str) -> int:
            return len(tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"])

        def runner(question: str, context: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
            enc = tokenizer(
                question, context, return_tensors="pt", return_offsets_mapping=True, truncation=False
            )
            offsets = enc.pop("offset_mapping")[0].numpy()
            context_mask = np.array([sid == 1 for sid in enc.sequence_ids(0)], dtype=bool)
            with torch.inference_mode():
                out = model(**enc.to(resolved_device))
            return (
                out.start_logits[0].float().cpu().numpy(),
                out.end_logits[0].float().cpu().numpy(),
                offsets,
                context_mask,
            )

        return cls(runner, count_tokens, resolved_device, source, _model=model, _tokenizer=tokenizer)

    def _validate(self, question: Any, context: Any, max_answer_tokens: Any) -> tuple[int, int]:
        question, context = _check_inputs(question, context, max_answer_tokens)
        return _check_token_counts(self._count_tokens(question), self._count_tokens(context))

    def answer(
        self, question: str, context: str, *, max_answer_tokens: int = DEFAULT_MAX_ANSWER_TOKENS
    ) -> dict[str, Any]:
        """Extract the best context span or the SQuAD 2.0 empty answer (upstream null-vs-span rule)."""
        n_question, n_context = self._validate(question, context, max_answer_tokens)
        start, end, offsets, context_mask = self._runner(question, context)
        start, end = np.asarray(start, dtype=np.float64), np.asarray(end, dtype=np.float64)
        context_mask = np.asarray(context_mask, dtype=bool)
        offsets = np.asarray(offsets)
        if not (start.shape == end.shape == context_mask.shape) or offsets.shape != (start.shape[0], 2):
            raise RuntimeError(
                f"runner returned inconsistent shapes {start.shape} {end.shape} {offsets.shape}"
            )
        allowed = context_mask.copy()
        allowed[0] = True  # <s> stays eligible so the null answer has a score
        start = np.where(allowed, start, NULL_MASK_VALUE)
        end = np.where(allowed, end, NULL_MASK_VALUE)
        p_start = np.exp(start - start.max())
        p_start /= p_start.sum()
        p_end = np.exp(end - end.max())
        p_end /= p_end.sum()
        no_answer_score = float(p_start[0] * p_end[0])
        p_start[0] = p_end[0] = 0.0
        outer = np.tril(np.triu(np.outer(p_start, p_end)), max_answer_tokens - 1)
        outer[~context_mask, :] = 0.0
        outer[:, ~context_mask] = 0.0
        best_index = int(np.argmax(outer))
        span_start, span_end = np.unravel_index(best_index, outer.shape)
        best_span_score = float(outer[span_start, span_end])
        if no_answer_score > best_span_score:
            answer_text, char_start, char_end, score = "", 0, 0, no_answer_score
        else:
            char_start, char_end = int(offsets[span_start][0]), int(offsets[span_end][1])
            answer_text, score = context[char_start:char_end], best_span_score
        return {
            "answer": answer_text,
            "score": score,
            "start": char_start,
            "end": char_end,
            "no_answer_score": no_answer_score,
            "best_span_score": best_span_score,
            "answerable": bool(answer_text),
            "question_tokens": n_question,
            "context_tokens": n_context,
            "max_answer_tokens": max_answer_tokens,
            "decision_rule": DECISION_RULE,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def check_fit(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Split validated records into those within the token ceilings and those `answer` would refuse."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_RECORDS_FIT)["records"]
        fitting, dropped = [], []
        for record in checked:
            if (
                self._count_tokens(record["question"]) <= MAX_QUESTION_TOKENS
                and self._count_tokens(record["context"]) <= MAX_CONTEXT_TOKENS
            ):
                fitting.append(record)
            else:
                dropped.append(record["id"])
        return {"fitting": fitting, "dropped": dropped, "n_fitting": len(fitting), "n_dropped": len(dropped)}

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, max_answer_tokens: int = DEFAULT_MAX_ANSWER_TOKENS
    ) -> dict[str, Any]:
        """Answer every record and score the predictions against its gold list (corpus exact-match and F1)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import qa_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions = []
        for record in checked:
            predictions.append(
                self.answer(record["question"], record["context"], max_answer_tokens=max_answer_tokens)[
                    "answer"
                ]
            )
        metrics = qa_metrics(predictions, [[a["text"] for a in r["answers"]] for r in checked])
        metrics.update(
            {
                "max_answer_tokens": max_answer_tokens,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _trainable_names(self, trainable_encoder_layers: int) -> list[str]:
        if (
            not isinstance(trainable_encoder_layers, int)
            or not 1 <= trainable_encoder_layers <= ENCODER_LAYERS
        ):
            raise ValueError(f"trainable_encoder_layers must be an int in 1..{ENCODER_LAYERS}")
        model, _ = self._require_model()
        first = ENCODER_LAYERS - trainable_encoder_layers
        prefixes = tuple(f"roberta.encoder.layer.{k}." for k in range(first, ENCODER_LAYERS)) + (
            "qa_outputs.",
        )
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def _span_positions(self, encoded: Any, index: int, record: Mapping[str, Any]) -> tuple[int, int]:
        """Token start/end of the first gold answer inside the context segment; (0, 0) for unanswerable."""
        if not record["answers"]:
            return 0, 0
        answer = record["answers"][0]
        char_start, char_end = int(answer["answer_start"]), int(answer["answer_start"]) + len(answer["text"])
        offsets = encoded["offset_mapping"][index].tolist()
        sequence_ids = encoded.sequence_ids(index)
        start = end = None
        for pos, (sid, (o_start, o_end)) in enumerate(zip(sequence_ids, offsets, strict=True)):
            if sid != 1 or o_end <= o_start:
                continue
            if start is None and o_end > char_start:
                start = pos
            if o_start < char_end:
                end = pos
        if start is None or end is None or end < start:
            raise ValueError(f"record {record['id']}: gold span does not map onto context tokens")
        return start, end

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 2,
        lr: float = 3e-5,
        batch_size: int = 16,
        trainable_encoder_layers: int = DEFAULT_TRAINABLE_ENCODER_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning on a validated QA dataset.

        Only the last `trainable_encoder_layers` encoder blocks and the span head train (2 blocks by
        default: 14,177,282 of 124,056,578 parameters; the embeddings and the earlier blocks stay frozen).
        Start/end cross-entropy on the first gold span (position 0, `<s>`, for an unanswerable record —
        the SQuAD 2.0 convention), AdamW at a fixed learning rate with gradient clipping at 1.0, no
        truncation: every training record must already fit the ceilings (`check_fit`). Epoch 0 records the
        frozen model's validation F1; the epoch with the highest validation F1 is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        names = self._trainable_names(trainable_encoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        for record in train_checked:
            self._validate(record["question"], record["context"], DEFAULT_MAX_ANSWER_TOKENS)
        import torch

        torch.manual_seed(seed)
        model, tokenizer = self._require_model()
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = torch.device(self.device)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                k: v
                for k, v in self.evaluate(val_checked).items()
                if k in ("exact_match", "f1", "n", "answered_rate")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_f1 = entry["val"]["f1"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        for epoch in range(1, epochs + 1):
            model.train()
            order = torch.randperm(len(train_checked), generator=generator).tolist()
            losses = []
            for start in range(0, len(order), batch_size):
                batch = [train_checked[i] for i in order[start : start + batch_size]]
                encoded = tokenizer(
                    [r["question"] for r in batch],
                    [r["context"] for r in batch],
                    return_tensors="pt",
                    padding=True,
                    truncation=False,
                    return_offsets_mapping=True,
                )
                positions = [self._span_positions(encoded, i, r) for i, r in enumerate(batch)]
                encoded.pop("offset_mapping")
                out = model(
                    input_ids=encoded["input_ids"].to(device),
                    attention_mask=encoded["attention_mask"].to(device),
                    start_positions=torch.tensor([s for s, _e in positions], device=device),
                    end_positions=torch.tensor([e for _s, e in positions], device=device),
                )
                optimiser.zero_grad(set_to_none=True)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimiser.step()
                losses.append(float(out.loss.detach()))
            model.eval()
            entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
            history.append(entry)
            if progress:
                progress(entry)
            current = entry["val"]["f1"] if entry["val"] else math.inf
            if current > best_f1 or not entry["val"]:
                best_f1 = current
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                best_epoch = epoch
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_encoder_layers": trainable_encoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation F1" if val_checked else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted encoder-block and span-head tensors as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest and digest, then overwrite exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        entry = manifest["files"][0]
        weights_path = root / entry["path"]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != manifest["tensors"]:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not (
                key.startswith("roberta.encoder.layer.") or key.startswith("qa_outputs.")
            ):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable encoder or span-head tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> RoBERTaQuestionAnsweringPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/roberta_question_answering_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Extractive-QA dataset contract for fine-tuning: the pinned AdversarialQA (dRoBERTa) sample, validation,
seeded context-disjoint splitting, BYOD loaders and CSV export.

The default dataset is **real**: AdversarialQA v1.0 (Bartolo et al., TACL 2020, CC BY-SA 3.0) — SQuAD-style
question/answer pairs over Wikipedia passages, written by annotators who could see a model's predictions and
kept only the questions that model got wrong. The `3_droberta` subset was collected against a RoBERTa reader,
so it is the hardest of the three for this pipeline's `roberta-base-squad2` checkpoint: the frozen model's
exact-match/F1 here sits far below its SQuAD 2.0 dev figures, which is what makes it an honest adaptation
target. One 9.0 MB zip is fetched from the AdversarialQA site, refused on any byte-size or SHA-256 mismatch,
and two members are read without extracting to disk: `3_droberta/train.json` (10,000 questions, the
training pool) and `3_droberta/dev.json` (1,000 questions over 21 articles, split here by article into
validation and test). The `test.json` member ships without answers and is not used.

A record is ``{id, question, context, answers}`` with ``answers`` a list of ``{text, answer_start}`` (the
SQuAD convention; an empty list means unanswerable — AdversarialQA has none, so the null answer scores 0).
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_CONTEXT_CHARS, MAX_QUESTION_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "AdversarialQA"
CORPUS_RELEASE = "v1.0 (2020-09-23)"
CORPUS_URL = "https://adversarialqa.github.io/data/aqa_v1.0.zip"
CORPUS_BYTES = 9_018_914
CORPUS_SHA256 = "f4f3c23224a5060b28c35e35581bd5cf46256dda3665418fb83d036d0e0c93cf"
CORPUS_MEMBERS = {"train": "3_droberta/train.json", "dev": "3_droberta/dev.json"}
CORPUS_LICENSE = "CC BY-SA 3.0 (Bartolo et al. 2020; adversarialqa.github.io)"
CORPUS_QUESTIONS = {"train": 10_000, "dev": 1_000}
DEFAULT_CACHE_DIR = Path("weights") / "adversarialqa"
# Sample filters: the pipeline refuses pairs over MAX_QUESTION_TOKENS (64) / MAX_CONTEXT_TOKENS (384) with the
# real tokenizer, so the sample keeps passages short enough that none is refused (about 4 chars per BPE
# token on English Wikipedia text; the build record found 0 of the sampled pairs over either ceiling).
MAX_SAMPLE_CONTEXT_CHARS = 1_300
MAX_SAMPLE_QUESTION_CHARS = 200
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 1_000, "validation": 200, "test": 400}
MIN_RECORDS = 8
MAX_RECORDS = 20_000
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> bytes:
    """Return the pinned AdversarialQA zip bytes from the cache or the project site, digest-verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / "aqa_v1.0.zip"
    if local.is_file():
        data = local.read_bytes()
        if len(data) == CORPUS_BYTES and _sha256_bytes(data) == CORPUS_SHA256:
            return data
    if fetcher is not None:
        data = fetcher(CORPUS_URL)
    else:
        with urllib.request.urlopen(CORPUS_URL, timeout=180) as response:  # noqa: S310 (pinned https URL)
            data = response.read()
    if len(data) != CORPUS_BYTES or _sha256_bytes(data) != CORPUS_SHA256:
        raise ValueError(
            f"corpus zip: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
            f"pinned {CORPUS_BYTES} / {CORPUS_SHA256[:16]}…"
        )
    local.write_bytes(data)
    return data


def flatten_squad(data: Mapping[str, Any], *, prefix: str = "") -> list[dict[str, Any]]:
    """SQuAD-format ``{"data": [{"title", "paragraphs": [{"context", "qas": [...]}]}]}`` → flat records
    carrying the article ``title`` (used for article-disjoint splitting)."""
    if not isinstance(data, Mapping) or "data" not in data:
        raise ValueError("SQuAD-format JSON must be an object with a 'data' list")
    if not isinstance(data, Mapping) or "data" not in data:
        raise ValueError("SQuAD-format JSON must be an object with a 'data' list")
    records = []
    for article in data["data"]:
        title = str(article.get("title", ""))
        for paragraph in article["paragraphs"]:
            context = str(paragraph["context"])
            for qa in paragraph["qas"]:
                records.append(
                    {
                        "id": f"{prefix}{qa['id']}",
                        "question": str(qa["question"]),
                        "context": context,
                        "answers": [
                            {"text": str(a["text"]), "answer_start": int(a["answer_start"])}
                            for a in qa.get("answers", [])
                        ],
                        "title": title,
                    }
                )
    return records


def read_corpus(data: bytes) -> dict[str, list[dict[str, Any]]]:
    """The dRoBERTa train and dev members as flat records, read without extracting to disk."""
    archive = zipfile.ZipFile(io.BytesIO(data))
    names = set(archive.namelist())
    for member in CORPUS_MEMBERS.values():
        if member not in names:
            raise ValueError(f"corpus zip is missing member {member}")
    out = {}
    for split, member in CORPUS_MEMBERS.items():
        out[split] = flatten_squad(json.loads(archive.read(member).decode("utf-8")), prefix=f"{split}-")
        if len(out[split]) != CORPUS_QUESTIONS[split]:
            raise ValueError(f"{member}: {len(out[split])} questions, expected {CORPUS_QUESTIONS[split]}")
    return out


def filter_records(records: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Keep answerable records whose passage and question fit the sample character filters and whose
    every answer is found at its offset; drop exact duplicate (context, question) pairs."""
    seen: set[tuple[str, str]] = set()
    kept = []
    for record in records:
        context, question = str(record["context"]), str(record["question"])
        if len(context) > MAX_SAMPLE_CONTEXT_CHARS or len(question) > MAX_SAMPLE_QUESTION_CHARS:
            continue
        answers = list(record.get("answers", []))
        if not answers or not all(_answer_at_offset(context, a) for a in answers):
            continue
        key = (context.lower(), question.lower())
        if key in seen:
            continue
        seen.add(key)
        kept.append(dict(record))
    return kept


def _answer_at_offset(context: str, answer: Mapping[str, Any]) -> bool:
    start, text = int(answer["answer_start"]), str(answer["text"])
    return bool(text) and context[start : start + len(text)] == text


def _group_by(records: Sequence[Mapping[str, Any]], key: str) -> dict[str, list[dict[str, Any]]]:
    groups: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        groups.setdefault(str(record.get(key, "")).lower(), []).append(dict(record))
    return groups


def build_sample_dataset(
    corpus: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Training records: a seeded subset of the filtered dRoBERTa training pool. Validation and test: the
    filtered dev questions cut **by article** (a seeded shuffle of article titles fills validation first), so
    no passage — and no article — is shared between validation and test; the training pool's articles are
    disjoint from dev by AdversarialQA's construction, which `check_split_disjoint` re-checks on passages."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    train_pool = filter_records(corpus["train"])
    if sizes["train"] > len(train_pool):
        raise ValueError(f"requested {sizes['train']} training records but only {len(train_pool)} fit")
    rng.shuffle(train_pool)
    dev_pool = filter_records(corpus["dev"])
    groups = _group_by(dev_pool, "title")
    titles = sorted(groups)
    rng.shuffle(titles)
    validation: list[dict[str, Any]] = []
    test: list[dict[str, Any]] = []
    for title in titles:
        (validation if len(validation) < sizes["validation"] else test).extend(groups[title])
    if len(test) < sizes["test"] or len(validation) < sizes["validation"]:
        raise ValueError(
            f"dev pool yields {len(validation)} validation / {len(test)} test records; "
            f"requested {sizes['validation']} / {sizes['test']}"
        )
    rng.shuffle(validation)
    rng.shuffle(test)
    splits = {
        "train": train_pool[: sizes["train"]],
        "validation": validation[: sizes["validation"]],
        "test": test[: sizes["test"]],
    }
    return {
        name: [
            {
                "id": f"{name}-{i:04d}",
                "question": r["question"],
                "context": r["context"],
                "answers": r["answers"],
                "title": r.get("title", ""),
            }
            for i, r in enumerate(part)
        ]
        for name, part in splits.items()
    }


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/question/context/answers")
    for key in ("id", "question", "context", "answers"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid, question, context, answers = record["id"], record["question"], record["context"], record["answers"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    for key, value, ceiling in (
        ("question", question, MAX_QUESTION_CHARS),
        ("context", context, MAX_CONTEXT_CHARS),
    ):
        if not isinstance(value, str):
            raise ValueError(f"{label}: {key} must be a string")
        if not value.strip():
            raise ValueError(f"{label}: {key} is empty")
        if len(value) > ceiling:
            raise ValueError(
                f"{label}: {key} has {len(value)} chars; ceiling is MAX_{key.upper()}_CHARS={ceiling}"
            )
    if isinstance(answers, Mapping) or not isinstance(answers, Sequence) or isinstance(answers, (str, bytes)):
        raise ValueError(
            f"{label}: answers must be a list of {{text, answer_start}} (empty for unanswerable)"
        )
    checked_answers = []
    for j, answer in enumerate(answers):
        if not isinstance(answer, Mapping) or "text" not in answer or "answer_start" not in answer:
            raise ValueError(f"{label}: answers[{j}] must be a mapping with text and answer_start")
        start, text = answer["answer_start"], answer["text"]
        if isinstance(start, bool) or not isinstance(start, int) or not isinstance(text, str):
            raise ValueError(f"{label}: answers[{j}] needs an int answer_start and a str text")
        if not _answer_at_offset(context, answer):
            raise ValueError(f"{label}: answers[{j}] text {text[:40]!r} is not found at offset {start}")
        checked_answers.append({"text": text, "answer_start": start})
    item = {"id": rid, "question": question, "context": context, "answers": checked_answers}
    if "title" in record:
        item["title"] = str(record["title"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a QA dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, question, context, answers} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    contexts: set[str] = set()
    unanswerable = 0
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        contexts.add(item["context"].lower())
        unanswerable += not item["answers"]
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_contexts": len(contexts),
        "unanswerable": unanswerable,
        "question_chars": {
            "min": min(len(r["question"]) for r in checked),
            "max": max(len(r["question"]) for r in checked),
        },
        "context_chars": {
            "min": min(len(r["context"]) for r in checked),
            "max": max(len(r["context"]) for r in checked),
        },
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [
        [r["id"], r["question"], r["context"], [[a["text"], a["answer_start"]] for a in r["answers"]]]
        for r in records
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def gold_texts(record: Mapping[str, Any]) -> list[str]:
    """The SQuAD-style gold list for the metric helpers (empty for unanswerable)."""
    return [a["text"] for a in record["answers"]]


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no lower-cased passage appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["context"]).lower()
            if key in seen and seen[key] != name:
                raise ValueError(
                    f"a passage ({record['context'][:60]!r}…) appears in both {seen[key]} and {name}"
                )
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded split of a BYOD dataset into train/validation/test **by passage**: every question on the
    same passage lands in the same split, so a test passage is never seen in training."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    groups = list(_group_by(checked, "context").values())
    random.Random(seed).shuffle(groups)
    n_test = max(1, round(len(checked) * test_fraction))
    n_val = round(len(checked) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for group in groups:
        if len(splits["test"]) < n_test:
            splits["test"].extend(group)
        elif len(splits["validation"]) < n_val:
            splits["validation"].extend(group)
        else:
            splits["train"].extend(group)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a JSON array of ``{id, question, context, answers}``, a SQuAD-format JSON file
    (``{"data": [...]}``), JSONL, or a CSV with columns ``id, question, context, answer_text, answer_start``
    (one answer per row; an empty ``answer_text`` means unanswerable)."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".csv":
        rows = list(csv.DictReader(io.StringIO(text)))
        missing = {"id", "question", "context", "answer_text", "answer_start"} - set(
            rows[0].keys() if rows else set()
        )
        if missing:
            raise ValueError(f"CSV is missing columns {sorted(missing)}")
        return [
            {
                "id": r["id"],
                "question": r["question"],
                "context": r["context"],
                "answers": (
                    [{"text": r["answer_text"], "answer_start": int(r["answer_start"])}]
                    if r["answer_text"]
                    else []
                ),
            }
            for r in rows
        ]
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if isinstance(data, Mapping) and "data" in data:
            return flatten_squad(data)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records or a SQuAD-format object")
        return data
    raise ValueError("BYOD datasets must be .csv, .json or .jsonl")


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One row per record with its first gold answer (empty for unanswerable)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle, fieldnames=["id", "question", "context", "answer_text", "answer_start"]
        )
        writer.writeheader()
        for record in records:
            first = record["answers"][0] if record["answers"] else {"text": "", "answer_start": ""}
            writer.writerow(
                {
                    "id": record["id"],
                    "question": record["question"],
                    "context": record["context"],
                    "answer_text": first["text"],
                    "answer_start": first["answer_start"],
                }
            )
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `7`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `adc3b06f79f7…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `RoBERTaQuestionAnsweringPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "roberta-base-squad2",
  "modelId": "deepset/roberta-base-squad2",
  "revision": "adc3b06f79f797d1c575d5479d6f5efe54a9e3b4",
  "files": [
    {
      "path": "README.md",
      "bytes": 9181,
      "sha256": "04834cd9007b2cdd1b1bdd501b571579577331e467be6e24e47d40f060558a29"
    },
    {
      "path": "config.json",
      "bytes": 571,
      "sha256": "64fa58495a722d57609c22f199824bfe98c19be068136a70c268214a08cb8060"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 496254442,
      "sha256": "ac5db66fdcfecb400345d09787b71009d60805ef9883451071669cf951b5e2c7"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 772,
      "sha256": "c611b1f7d416eb001ee4f293d903ea8c88e703463f1d403f1866a0352743fd00"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 79,
      "sha256": "7a33226d4265e3989cc6341666af179d0cc710136f4059aae0dd8c0797cba556"
    },
    {
      "path": "vocab.json",
      "bytes": 898822,
      "sha256": "06b4d46c8e752d410213d9548eb27a54db70fda0319b6271fb8d59dead5e1cab"
    }
  ],
  "totalBytes": 497620185
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = RoBERTaQuestionAnsweringPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Adversarial QA corpus, validation and split

`fetch_corpus` downloads the pinned AdversarialQA zip (or reads it from the cache), refuses a byte-size or SHA-256 mismatch before the archive is opened, and `read_corpus` reads the two dRoBERTa members without extracting to disk, flattening the SQuAD structure into flat records that keep their article title. `build_sample_dataset` keeps answerable records whose passage is at most 1,300 characters and whose every gold span is found at its offset, drops exact duplicate pairs, draws 1,000 training questions from the 10,000-question training pool by a seeded shuffle, and cuts the 1,000-question dev member **by article** into 200 validation and 400 test questions, so no passage and no article is shared between validation and test (the training pool's articles are disjoint from dev by AdversarialQA's construction). `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no passage appears in two splits, and the training split is written to `outputs/roberta_question_answering_train.csv` in the shape BYOD expects.

Look for: 10,000 + 1,000 raw questions, three digests, splits 1,000 / 200 / 400 over 358 / 5 / 16 articles, zero unanswerable records, and four refusal probes — a duplicate id, a gold span at the wrong offset, a missing field and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_questions = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/adversarialqa'))
    raw_questions = {name: len(part) for name, part in corpus.items()}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} {CORPUS_RELEASE} dRoBERTa subset ({CORPUS_LICENSE})'
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
articles = {name: len({r.get('title', '') for r in part}) for name, part in splits.items()}
write_dataset_csv(splits['train'], 'outputs/roberta_question_answering_train.csv')
print({'data_source': data_source, 'raw_questions': raw_questions, 'splits': disjoint, 'articles': articles, 'corpus_sha256': CORPUS_SHA256[:16] + '...'})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_contexts': manifest['unique_contexts'], 'unanswerable': manifest['unanswerable'], 'context_chars': manifest['context_chars'], 'digest': manifest['digest'][:16] + '...'}})
example = splits['train'][0]
print({'example': {'id': example['id'], 'question': example['question'], 'answers': example['answers'], 'context': example['context'][:160] + '...'}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in splits['train'][:8]],
    'gold span at the wrong offset': [{**splits['train'][0], 'answers': [{'text': splits['train'][0]['answers'][0]['text'], 'answer_start': 0}]}, *splits['train'][1:8]],
    'missing field': [{'id': r['id'], 'question': r['question'], 'context': r['context']} for r in splits['train'][:8]],
    'too small': splits['train'][:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Fit check, then answer through the inference contract

The token ceilings `MAX_QUESTION_TOKENS` (64) and `MAX_CONTEXT_TOKENS` (384, the upstream fine-tuning window) need the real tokenizer, so they are applied now that the model is loaded: `pipe.check_fit` partitions each split into the records `answer` accepts and the ones it would **refuse with a `ValueError` naming the count, never silently truncate or window**. The dropped ids are printed and the fitting records are what every later cell uses (the build record dropped 1 of 1,600 sample records — a 1,295-character passage that tokenises to 460 pieces).

Then the inference contract is exercised as it always was on three authored questions over one two-sentence passage — two answerable, one deliberately unanswerable. `validate_inputs` applies exactly the checks `answer` applies (text types, non-emptiness, the character ceilings, `max_answer_tokens` in 1..`MAX_ANSWER_TOKENS`) and returns an input manifest; an out-of-range setting is validated too and its rejection recorded as a finding. `answer` returns the extracted span (or `''` for no-answer), character offsets, `score`, `no_answer_score`, `best_span_score`, `answerable`, both token counts and the model identity. **Score semantics:** every score is a **product of two softmax masses** normalised over the passage — a ranking signal that shrinks as the passage grows, **not a calibrated probability** — and no no-answer threshold offset ships. Whether the answers are *right* is what Section 6 measures on 400 gold spans, not what three authored pairs can tell you.

In [ ]:
import time

ANSWER_MAX_TOKENS = 15  # @param {type:"integer"}

fit = {name: pipe.check_fit(part) for name, part in splits.items()}
train_records, val_records, test_records = fit['train']['fitting'], fit['validation']['fitting'], fit['test']['fitting']
print({'fit_check': {name: {'fitting': f['n_fitting'], 'dropped': f['dropped']} for name, f in fit.items()}})
ceilings = {'MAX_QUESTION_CHARS': MAX_QUESTION_CHARS, 'MAX_CONTEXT_CHARS': MAX_CONTEXT_CHARS, 'MAX_QUESTION_TOKENS': MAX_QUESTION_TOKENS, 'MAX_CONTEXT_TOKENS': MAX_CONTEXT_TOKENS, 'MAX_ANSWER_TOKENS': MAX_ANSWER_TOKENS, 'DEFAULT_MAX_ANSWER_TOKENS': DEFAULT_MAX_ANSWER_TOKENS}
print(ceilings)
print({'decision_rule': DECISION_RULE})

passage = (
    'The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. '
    'It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889.'
)
questions = ['Who designed the Eiffel Tower?', 'When was the tower built?', 'What colour is the tower painted?']
contexts = [passage] * len(questions)
golds = [['Gustave Eiffel'], ['1887 to 1889', 'from 1887 to 1889'], []]
item_ids = [f'pair{index:02d}' for index in range(len(questions))]
input_manifest = validate_inputs(questions, contexts, max_answer_tokens=ANSWER_MAX_TOKENS, names=item_ids)
try:
    validate_inputs(questions, contexts, max_answer_tokens=MAX_ANSWER_TOKENS + 1)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'max-answer-tokens-ceiling-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/roberta_question_answering_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
results = []
for item_id, question, context in zip(item_ids, questions, contexts, strict=True):
    started = time.perf_counter()
    result = pipe.answer(question, context, max_answer_tokens=ANSWER_MAX_TOKENS)
    results.append({'id': item_id, 'question': question, 'context': context, 'seconds': round(time.perf_counter() - started, 3), **result})
    print(f"{item_id} [{'answerable' if result['answerable'] else 'no answer'}] score={result['score']:.4f} null={result['no_answer_score']:.4f} best_span={result['best_span_score']:.4f} {result['question_tokens']}+{result['context_tokens']} tokens: {result['answer']!r} [{result['start']}:{result['end']}]")
checks = {
    'one_result_per_pair': len(results) == len(questions),
    'span_is_substring_at_offsets': all(r['context'][r['start']:r['end']] == r['answer'] for r in results),
    'question_within_ceiling': all(r['question_tokens'] <= MAX_QUESTION_TOKENS for r in results),
    'context_within_ceiling': all(r['context_tokens'] <= MAX_CONTEXT_TOKENS for r in results),
    'scores_in_unit_interval': all(0.0 <= r[k] <= 1.0 for r in results for k in ('score', 'no_answer_score', 'best_span_score')),
    'null_rule_consistent': all(r['answerable'] == (r['no_answer_score'] <= r['best_span_score']) for r in results),
    'setting_echoed': all(r['max_answer_tokens'] == ANSWER_MAX_TOKENS for r in results),
}
if not all(checks.values()):
    raise RuntimeError(f'answer output failed a sanity check: {checks}')
print({'checks': checks, 'unanswered': [r['id'] for r in results if not r['answerable']], 'findings': len(input_manifest['findings'])})

## 6. Baselines and the frozen model's score on the test split

Three numbers frame the adaptation. The **always-null baseline** returns the empty answer for every question and scores exactly the unanswerable fraction of the set — zero here, which is the point: on AdversarialQA every null answer the frozen model returns is a miss. The **lexical-overlap baseline** returns the passage sentence sharing the most normalised tokens with the question — a bag-of-words reader with no model, whose F1 comes from partial overlap with long spans. The **frozen model** answers the 400 test questions with the `max_answer_tokens` from Section 5 and is scored with the same two metrics: corpus **exact-match** and **F1** with the official SQuAD 2.0 normalisation (lower-case, punctuation and articles stripped; an empty gold matches only an empty prediction). Read `answered_rate` beside them: the fraction of questions for which a span was returned at all. Expect the frozen F1 to be low — these questions were selected because a RoBERTa reader failed them — and expect the null answer for roughly half of the test questions, a systematic under-answering that the adaptation in Section 7 is meant to correct.

In [ ]:
baseline_null = null_baseline(test_records)
baseline_lexical = lexical_overlap_baseline(test_records)
print({'always_null_baseline': {'exact_match': round(baseline_null['exact_match'], 2), 'f1': round(baseline_null['f1'], 2), 'n': baseline_null['n']}})
print({'lexical_overlap_baseline': {'exact_match': round(baseline_lexical['exact_match'], 2), 'f1': round(baseline_lexical['f1'], 2), 'n': baseline_lexical['n']}})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, max_answer_tokens=ANSWER_MAX_TOKENS)
print({'frozen_model_test': {'exact_match': round(frozen_test['exact_match'], 2), 'f1': round(frozen_test['f1'], 2), 'answered_rate': round(frozen_test['answered_rate'], 1), 'n': frozen_test['n'], 'verdict': frozen_test['verdict']}, 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
for record in test_records[:3]:
    item = pipe.answer(record['question'], record['context'], max_answer_tokens=ANSWER_MAX_TOKENS)
    print({'question': record['question'], 'frozen': item['answer'], 'gold': gold_texts(record)})
assert frozen_test['f1'] > baseline_null['f1']

## 7. Bounded fine-tuning of the last encoder blocks and the span head

`pipe.adapt` trains only the last `TRAINABLE_ENCODER_LAYERS` encoder blocks plus the span head `qa_outputs` — two blocks by default, 14,177,282 of 124,056,578 parameters; the embeddings and the earlier blocks stay frozen — with start/end cross-entropy on the first gold span (position 0, `<s>`, for an unanswerable record, the SQuAD 2.0 convention), AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Nothing is truncated: every training record passed the fit check. Epoch 0 records the frozen model's validation exact-match and F1; every epoch is scored on the validation split, and the epoch with the highest validation F1 is kept.

Watch validation F1 roughly double in the first epoch and the answered rate jump to 100 % (about 3.5 minutes per epoch on CPU, validation scoring included). The build record's counter-examples are in the model card; the default is the smallest configuration that captured most of the gain.

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 3e-5  # @param {type:"number"}
BATCH_SIZE = 16  # @param {type:"integer"}
TRAINABLE_ENCODER_LAYERS = 2  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_exact_match'] = round(entry['val']['exact_match'], 2)
        row['val_f1'] = round(entry['val']['f1'], 2)
        row['val_answered_rate'] = round(entry['val']['answered_rate'], 1)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_encoder_layers=TRAINABLE_ENCODER_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or epoch selection, and none of its passages or articles appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6, and the four numbers are put side by side. Look for an F1 gain of ten points or more and an answered rate at or near 100 % — the cell asserts the adapted F1 is above the frozen F1 — and for the same three questions answered by the adapted model. Four hundred questions from one seeded split of one corpus give no dispersion estimate; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on adversarial Wikipedia questions says nothing about your documents until you measure it there. Note what the adaptation also removes: the model now answers every question, so its ability to say *no answer* on SQuAD 2.0-style unanswerable questions is traded away on this corpus, which contains none.

In [ ]:
adapted_test = pipe.evaluate(test_records, max_answer_tokens=ANSWER_MAX_TOKENS)
adapted_val = pipe.evaluate(val_records, max_answer_tokens=ANSWER_MAX_TOKENS)
comparison = {
    'exact_match': {'always_null': round(baseline_null['exact_match'], 2), 'lexical_overlap': round(baseline_lexical['exact_match'], 2), 'frozen': round(frozen_test['exact_match'], 2), 'adapted': round(adapted_test['exact_match'], 2)},
    'f1': {'always_null': round(baseline_null['f1'], 2), 'lexical_overlap': round(baseline_lexical['f1'], 2), 'frozen': round(frozen_test['f1'], 2), 'adapted': round(adapted_test['f1'], 2)},
    'answered_rate': {'frozen': round(frozen_test['answered_rate'], 1), 'adapted': round(adapted_test['answered_rate'], 1)},
    'delta_vs_frozen': {'exact_match': round(adapted_test['exact_match'] - frozen_test['exact_match'], 2), 'f1': round(adapted_test['f1'] - frozen_test['f1'], 2)},
}
for metric, row in comparison.items():
    print({metric: row})
for record in test_records[:3]:
    item = pipe.answer(record['question'], record['context'], max_answer_tokens=ANSWER_MAX_TOKENS)
    print({'question': record['question'], 'adapted': item['answer'], 'gold': gold_texts(record)})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'fit_check': {name: {'fitting': f['n_fitting'], 'dropped': f['dropped']} for name, f in fit.items()},
    'max_answer_tokens': ANSWER_MAX_TOKENS,
    'baselines': {'always_null': baseline_null, 'lexical_overlap': baseline_lexical},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/roberta_question_answering_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['f1'] > frozen_test['f1']
print({'report': 'outputs/roberta_question_answering_evaluation_report.json'})

## 9. Answer new questions, export the adapter and reload it

Six questions from dev articles that were in none of the splits (they were filtered out of the sample by the passage-length filter, so they are also a small look at longer passages) are answered by the adapted model through the same `answer` contract as Section 5 and scored with `pipe.evaluate`, which returns a `measured-small-sample` verdict because six questions carry no dispersion estimate; the single-pair `evaluation_report` helper is written for the first of them, as the inference-only tutorial did.

`pipe.save_artifact` writes the trained tensors — the last two encoder blocks and the span head, about 57 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `RoBERTaQuestionAnsweringPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not an adaptable encoder-block or span-head tensor, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical answers (VER4).

In [ ]:
import csv
import shutil

if USE_BYOD:
    new_records = [{**r, 'id': f'new-{i:02d}'} for i, r in enumerate(test_records[:6])]
else:
    used = {r['context'].lower() for part in splits.values() for r in part}
    candidates = [r for r in corpus['dev'] if r['context'].lower() not in used and r['answers'] and len(r['context']) <= MAX_CONTEXT_CHARS]
    new_records = pipe.check_fit([{**r, 'id': f'new-{i:02d}'} for i, r in enumerate(candidates[:12])])['fitting'][:6]
new_metrics = pipe.evaluate(new_records, max_answer_tokens=ANSWER_MAX_TOKENS)
new_results = []
for record in new_records:
    item = pipe.answer(record['question'], record['context'], max_answer_tokens=ANSWER_MAX_TOKENS)
    new_results.append({'id': record['id'], 'question': record['question'], 'answer': item['answer'], 'gold': gold_texts(record), 'score': item['score'], 'no_answer_score': item['no_answer_score'], 'answerable': item['answerable'], 'question_tokens': item['question_tokens'], 'context_tokens': item['context_tokens']})
    print({k: new_results[-1][k] for k in ('id', 'question', 'answer', 'gold')})
single_report = evaluation_report({'answer': new_results[0]['answer']}, new_results[0]['gold'], sample_kind='one unseen AdversarialQA pair' if not USE_BYOD else 'one BYOD test record')
print({'new_questions': {'n': new_metrics['n'], 'exact_match': round(new_metrics['exact_match'], 2), 'f1': round(new_metrics['f1'], 2), 'verdict': new_metrics['verdict']}, 'single_pair_report_verdict': single_report['verdict']})
with open('outputs/roberta_question_answering_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['id', 'question', 'answer', 'gold', 'score', 'no_answer_score', 'answerable', 'question_tokens', 'context_tokens'])
    writer.writeheader()
    for row in new_results:
        writer.writerow({**row, 'gold': ' | '.join(row['gold'])})

artifact_dir = Path('outputs/roberta_question_answering_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'roberta_question_answering', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = RoBERTaQuestionAnsweringPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [pipe.answer(r['question'], r['context'], max_answer_tokens=ANSWER_MAX_TOKENS)['answer'] for r in test_records[:8]]
after = [reloaded.answer(r['question'], r['context'], max_answer_tokens=ANSWER_MAX_TOKENS)['answer'] for r in test_records[:8]]
parity = {'identical_answers': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_answers'] == parity['of']

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'model_attribution': 'deepset (https://huggingface.co/deepset/roberta-base-squad2), weights redistributed unmodified under CC-BY-4.0',
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'url': CORPUS_URL, 'sha256': CORPUS_SHA256, 'license': CORPUS_LICENSE, 'members': CORPUS_MEMBERS},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'items': [{k: r[k] for k in ('id', 'question', 'answer', 'start', 'end', 'score', 'no_answer_score', 'best_span_score', 'answerable', 'question_tokens', 'context_tokens', 'seconds')} for r in results], 'golds': golds},
    'comparison': comparison,
    'new_questions': new_metrics,
    'single_pair_report': single_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/roberta_question_answering_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen model fails most adversarial questions — its test F1 sits near the lexical-overlap baseline and it returns the empty answer for more than half of the answerable questions — and a bounded fine-tuning of the last two encoder blocks and the span head on 1,000 in-distribution questions roughly doubles held-out F1 and lifts the answered rate to 100 % in a few minutes on CPU, with a 57 MB adapter that reloads to identical answers. That is the claim: the adaptation contract works end to end on a real gold-span corpus, and the numbers it produces are read against two non-neural baselines and the frozen model rather than in isolation.

The test split is 400 questions over 16 articles from one seeded split of one corpus, the metrics are two reference-based scores (own implementations of the SQuAD 2.0 normalisation, and neither a human judgement), and AdversarialQA is Wikipedia prose with single-span gold answers and no unanswerable questions. So a gain here says the contract works, not that the adapted model is better on your documents, that it handles long or technical passages, or that its spans are faithful — a reader can return a plausible wrong span, and after this adaptation it answers every question, so the null decision the base model was trained for is degraded on this corpus and must be re-measured on unanswerable questions from your own domain before it is relied on. Fine-tuning on a narrow corpus can also erode the model elsewhere; nothing here measures that.

Three things to carry to real data. **Baselines first:** the always-null and lexical-overlap baselines and the frozen model's score on *your* gold spans are the numbers to read before any adapted one, separately for answerable and unanswerable questions. **Leakage:** keep every question on a passage in one split (the contract does this) and split by document or article when your questions come from one, never at random over near-duplicate passages. **Ceilings:** passages over `MAX_CONTEXT_TOKENS` are refused at inference and dropped by the fit check before training — long-document reading is out of scope and must be chunked by the caller.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real gold-span corpus, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against two trivial baselines and the frozen model on an article-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, reading-comprehension accuracy on any other domain, a usable no-answer threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_ENCODER_LAYERS = 1` and compare the artifact size and the test scores; raise `EPOCHS` and watch the validation F1 pick the epoch; add unanswerable records to a BYOD set (empty `answers`) and read the always-null baseline and the adapted `answered_rate` together; or bring your own documents through BYOD and read the two baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/roberta-squad2-question-answering-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/roberta-squad2-question-answering-pipeline/blob/main/MODEL_CARD.md
- Weight provenance and CC-BY-4.0 attribution: https://github.com/kurtvalcorza/roberta-squad2-question-answering-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model (deepset, CC-BY-4.0): https://huggingface.co/deepset/roberta-base-squad2
- Upstream code: https://github.com/deepset-ai/haystack
- RoBERTa: A Robustly Optimized BERT Pretraining Approach (Liu et al., 2019): https://arxiv.org/abs/1907.11692
- Know What You Don't Know: Unanswerable Questions for SQuAD (Rajpurkar et al., ACL 2018): https://arxiv.org/abs/1806.03822
- Beat the AI: Investigating Adversarial Human Annotation for Reading Comprehension (Bartolo et al., TACL 2020; AdversarialQA v1.0, CC BY-SA 3.0): https://arxiv.org/abs/2002.00293 — data: https://adversarialqa.github.io/
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)